# Solution A2 · Regression tree and feature importance from scratch

**Type**: Implement from scratch

## Objective

Implement a regression tree (squared error reduction, prediction = mean of the leaf) and compute the **impurity importance**: the sum, for each feature, of the impurity reductions of its splits weighted by the fraction of examples. Check that the importances match
`feature_importances_` of scikit-learn.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

data = load_diabetes()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, random_state=0)
sk = DecisionTreeRegressor(max_depth=4, min_samples_leaf=5, random_state=0).fit(X_tr, y_tr)

## Commented solution

In [ ]:
def best_split_reg(X, y, min_leaf):
    n, p = X.shape
    var0 = (y ** 2).mean() - y.mean() ** 2
    best = (0, None, None)
    for j in range(p):
        o = np.argsort(X[:, j], kind="stable")
        xs, ys = X[o, j], y[o]
        cs, cs2 = np.cumsum(ys), np.cumsum(ys ** 2)
        for i in range(min_leaf - 1, n - min_leaf):
            if xs[i] == xs[i + 1]:
                continue
            nl, nr = i + 1, n - i - 1
            var_left = cs2[i] / nl - (cs[i] / nl) ** 2
            var_right = (cs2[-1] - cs2[i]) / nr - ((cs[-1] - cs[i]) / nr) ** 2
            gain = var0 - (nl * var_left + nr * var_right) / n
            if gain > best[0] + 1e-12:
                best = (gain, j, (xs[i] + xs[i + 1]) / 2)
    return best

In [ ]:
def build_reg(X, y, max_depth, min_leaf, N, importances, depth=0):
    if depth == max_depth or len(y) < 2 * min_leaf or np.ptp(y) == 0:
        return {"leaf": y.mean()}
    gain, j, s = best_split_reg(X, y, min_leaf)
    if j is None:
        return {"leaf": y.mean()}
    importances[j] += gain * len(y) / N                                 # impurity reduction weighted by the share of examples of the node
    m = X[:, j] <= s
    return {"feature": j, "threshold": s,
            "left": build_reg(X[m], y[m], max_depth, min_leaf, N, importances, depth + 1),
            "right": build_reg(X[~m], y[~m], max_depth, min_leaf, N, importances, depth + 1)}


def predict_reg(tree, X):
    def one(x):
        node = tree
        while "leaf" not in node:
            node = node["left"] if x[node["feature"]] <= node["threshold"] else node["right"]
        return node["leaf"]
    return np.array([one(x) for x in X])

In [ ]:
importances = np.zeros(X_tr.shape[1])
tree = build_reg(X_tr, y_tr, 4, 5, len(y_tr), importances)
norm_importances = importances / importances.sum()
same_importances = bool(np.allclose(norm_importances, sk.feature_importances_))
train_agreement = float((np.abs(predict_reg(tree, X_tr) - sk.predict(X_tr)) < 1e-9).mean())
test_agreement = float((np.abs(predict_reg(tree, X_te) - sk.predict(X_te)) < 1e-9).mean())
top_feature = data.feature_names[int(norm_importances.argmax())]
top_two_share = float(np.sort(norm_importances)[-2:].sum())
print("importances:", dict(zip(data.feature_names, norm_importances.round(3))))
print(f"equal to scikit-learn's: {same_importances} | agreement on the predictions: training {train_agreement:.3f}, test {test_agreement:.3f} | top feature: {top_feature} | top two: {top_two_share:.1%}")

## Check

In [ ]:
with verify():
    g, f, s = best_split_reg(X_tr, y_tr, 5)
    check(f == sk.tree_.feature[0] and abs(s - sk.tree_.threshold[0]) < 1e-6 and g > 0, "the split at the root matches the one of scikit-learn", "variance with the cumulative sums of y and y^2")
    check(norm_importances is not None and close(norm_importances.sum(), 1.0) and same_importances is True, "normalized importances matching scikit-learn", "sum(gain * n_node / N) per feature, then divide by the sum")
    check(train_agreement == 1.0 and test_agreement >= 0.98, "equal predictions on the training set (100%) and on at least 98% of the test", "mean of the leaf; scikit-learn uses float32 for the thresholds")
    check(top_feature == data.feature_names[int(np.argmax(sk.feature_importances_))], "the most important feature matches the one of scikit-learn", "argmax of the importances")
    check(0.7 < top_two_share < 0.9 and (norm_importances[[3, 5]] < norm_importances[8]).all(), "the top two features are worth between 70% and 90% of the importance", "sum of the two largest importances")
done()

## What you learned

- **`feature_importances_` is an accounting**: for every split you compute how much impurity (here variance) it removed, weight it by the share of examples that go through that node and sum per feature. The two implementations match to the decimal: there is nothing else behind it.
- **The importances are relative to the tree:** they are normalized to sum 1 and depend on the depth and on the available features. The top two features (`s5`, `bmi`) are worth over 80%; with correlated features the importance is split arbitrarily among them.
- Regression reuses the scheme of classification with a different impurity: the **variance** (which is also the squared error of predicting the mean of the node). The leaf predicts the mean, and the gain is the reduction of the squared error.
- Remember the limits (D2 and lesson 7.3): the impurity importance is computed **on the training set**, it favors features with many distinct values and it does not say whether the feature helps out of sample.

## Common pitfall

Presenting `feature_importances_` as "the influence of the variables on the phenomenon". It is a statistic of the trained tree: it changes with the depth, with correlated features and with the sample.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How is feature importance computed in a tree?</summary>

For each feature I sum the impurity reduction of the splits that use it, weighted by the fraction of examples that go through the node, and normalize to sum 1 (in forests you average over the trees). It is fast but it has limits: it is computed on the training set, it favors features with many distinct values and it is split among correlated features.
For a more reliable judgment I use the permutation importance on validation data.

</details>